# 03 — Downtime Analysis

## Objective

Analyze downtime events before and after LSS to identify:

- total and average downtime
- downtime frequency
- main stop groups and stop reasons
- major downtime contributors using Pareto analysis
- changes in downtime patterns after LSS

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [2]:
downtime_before = pd.read_csv(RAW_DIR / "DowntimeDataset.csv")
downtime_after = pd.read_csv(RAW_DIR / "DowntimeDataset_afterLSS.csv")

downtime_before = (
    downtime_before
    .dropna(axis=1, how="all")
    .dropna(axis=0, how="all")
    .reset_index(drop=True)
)

downtime_after = (
    downtime_after
    .dropna(axis=1, how="all")
    .dropna(axis=0, how="all")
    .reset_index(drop=True)
)

C:\Users\erenk\AppData\Local\Temp\ipykernel_26408\3682822333.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  downtime_after = pd.read_csv(RAW_DIR / "DowntimeDataset_afterLSS.csv")


In [3]:
downtime_before = downtime_before.rename(columns={
    "Date": "date",
    "Productcode": "product_code",
    "StopGroup": "stop_group",
    "Stop": "stop",
    "StopType": "stop_type",
    "StopLocation": "stop_location",
    "ExtraText": "extra_text",
    "StopStartTime": "start_time",
    "StopEndTime": "end_time",
    "StopDuration(min)": "duration_min"
})

downtime_after = downtime_after.rename(columns={
    "Date": "date",
    "Product code": "product_code",
    "Stop group": "stop_group",
    "Stop": "stop",
    "Stop type": "stop_type",
    "Stop Location": "stop_location",
    "Extra text": "extra_text",
    "Stop start time": "start_time",
    "Stop end time": "end_time",
    "Stop duration (min) (TTR)": "duration_min"
})

In [4]:
downtime_before["date"] = pd.to_datetime(
    downtime_before["date"],
    format="%m/%d/%y"
)

downtime_after["date"] = pd.to_datetime(
    downtime_after["date"],
    format="%m/%d/%y"
)

downtime_before["period"] = "Before LSS"
downtime_after["period"] = "After LSS"

downtime = pd.concat(
    [downtime_before, downtime_after],
    ignore_index=True
)

In [5]:
downtime_summary = (
    downtime.groupby("period")
    .agg(
        events=("duration_min", "size"),
        production_days=("date", "nunique"),
        total_downtime_min=("duration_min", "sum"),
        avg_event_duration_min=("duration_min", "mean"),
        median_event_duration_min=("duration_min", "median")
    )
    .reset_index()
)

downtime_summary["events_per_day"] = (
    downtime_summary["events"]
    / downtime_summary["production_days"]
)

downtime_summary["downtime_min_per_day"] = (
    downtime_summary["total_downtime_min"]
    / downtime_summary["production_days"]
)

downtime_summary

,period,events,production_days,total_downtime_min,avg_event_duration_min,median_event_duration_min,events_per_day,downtime_min_per_day
0,After LSS,540,45,8710.0,16.12963,9.0,12.000000,193.555556
1,Before LSS,635,62,10387.8,16.35874,11.0,10.241935,167.545161


In [6]:
stop_type_summary = (
    downtime.groupby(["period", "stop_type"])
    .agg(
        events=("duration_min", "size"),
        total_downtime_min=("duration_min", "sum"),
        avg_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

stop_type_summary

,period,stop_type,events,total_downtime_min,avg_duration_min
0,After LSS,Planned - Included in OEE,109,1754.0,16.091743
1,After LSS,Planned - Not included in OEE,35,2835.0,81.000000
2,After LSS,Unplanned,396,4121.0,10.406566
3,Before LSS,Planned - Included in OEE,174,2850.9,16.384483
4,Before LSS,Planned - Not included in OEE,23,1351.7,58.769565
5,Before LSS,Unplanned,438,6185.2,14.121461


In [ ]:
unplanned = downtime[
    downtime["stop_type"].str.contains(
        "Unplanned",
        case=False,
        na=False
    )
]

unplanned_summary = (
    unplanned.groupby("period")
    .agg(
        events=("duration_min", "size"),
        active_days=("date", "nunique"),
        total_downtime_min=("duration_min", "sum"),
        avg_event_duration_min=("duration_min", "mean"),
        median_event_duration_min=("duration_min", "median")
    )
    .reset_index()
)

unplanned_summary

,period,events,active_days,total_downtime_min,avg_event_duration_min,median_event_duration_min
0,After LSS,396,43,4121.0,10.406566,7.0
1,Before LSS,438,62,6185.2,14.121461,9.0


In [8]:
unplanned_group_summary = (
    unplanned.groupby(["period", "stop_group"])
    .agg(
        events=("duration_min", "size"),
        total_downtime_min=("duration_min", "sum"),
        avg_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

unplanned_group_summary["downtime_share_pct"] = (
    unplanned_group_summary["total_downtime_min"]
    / unplanned_group_summary.groupby("period")["total_downtime_min"].transform("sum")
    * 100
)

unplanned_group_summary = unplanned_group_summary.sort_values(
    ["period", "total_downtime_min"],
    ascending=[True, False]
)

unplanned_group_summary

,period,stop_group,events,total_downtime_min,avg_duration_min,downtime_share_pct
1,After LSS,MATERIALS,258,2162.0,8.379845,52.462994
0,After LSS,FAILURE,116,1690.0,14.568966,41.009464
3,After LSS,SETUPS-CHANGEOVERS,11,135.0,12.272727,3.275904
2,After LSS,OPERATIONAL,11,134.0,12.181818,3.251638
4,Before LSS,FAILURE,222,3233.6,14.565766,52.279635
5,Before LSS,MATERIALS,189,2371.0,12.544974,38.333441
7,Before LSS,SETUPS-CHANGEOVERS,17,434.1,25.535294,7.018366
6,Before LSS,OPERATIONAL,10,146.5,14.650000,2.368557


In [9]:
for period in ["Before LSS", "After LSS"]:
    temp = (
        unplanned_group_summary[
            unplanned_group_summary["period"] == period
        ]
        .sort_values("total_downtime_min", ascending=False)
        .copy()
    )

    temp["cumulative_pct"] = temp["downtime_share_pct"].cumsum()

    display(temp)

,period,stop_group,events,total_downtime_min,avg_duration_min,downtime_share_pct,cumulative_pct
4,Before LSS,FAILURE,222,3233.6,14.565766,52.279635,52.279635
5,Before LSS,MATERIALS,189,2371.0,12.544974,38.333441,90.613076
7,Before LSS,SETUPS-CHANGEOVERS,17,434.1,25.535294,7.018366,97.631443
6,Before LSS,OPERATIONAL,10,146.5,14.650000,2.368557,100.000000


,period,stop_group,events,total_downtime_min,avg_duration_min,downtime_share_pct,cumulative_pct
1,After LSS,MATERIALS,258,2162.0,8.379845,52.462994,52.462994
0,After LSS,FAILURE,116,1690.0,14.568966,41.009464,93.472458
3,After LSS,SETUPS-CHANGEOVERS,11,135.0,12.272727,3.275904,96.748362
2,After LSS,OPERATIONAL,11,134.0,12.181818,3.251638,100.000000


In [10]:
production_days = {
    "Before LSS": 62,
    "After LSS": 43
}

group_comparison = unplanned_group_summary.copy()

group_comparison["production_days"] = (
    group_comparison["period"].map(production_days)
)

group_comparison["downtime_min_per_day"] = (
    group_comparison["total_downtime_min"]
    / group_comparison["production_days"]
)

group_comparison["events_per_day"] = (
    group_comparison["events"]
    / group_comparison["production_days"]
)

group_comparison[
    [
        "period",
        "stop_group",
        "downtime_min_per_day",
        "events_per_day",
        "avg_duration_min"
    ]
]

,period,stop_group,downtime_min_per_day,events_per_day,avg_duration_min
1,After LSS,MATERIALS,50.279070,6.000000,8.379845
0,After LSS,FAILURE,39.302326,2.697674,14.568966
3,After LSS,SETUPS-CHANGEOVERS,3.139535,0.255814,12.272727
2,After LSS,OPERATIONAL,3.116279,0.255814,12.181818
4,Before LSS,FAILURE,52.154839,3.580645,14.565766
5,Before LSS,MATERIALS,38.241935,3.048387,12.544974
7,Before LSS,SETUPS-CHANGEOVERS,7.001613,0.274194,25.535294
6,Before LSS,OPERATIONAL,2.362903,0.161290,14.650000


In [11]:
focus_stops = unplanned[
    unplanned["stop_group"].isin(["MATERIALS", "FAILURE"])
]

stop_reason_summary = (
    focus_stops.groupby(["period", "stop_group", "stop"])
    .agg(
        events=("duration_min", "size"),
        total_downtime_min=("duration_min", "sum"),
        avg_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

stop_reason_summary["production_days"] = (
    stop_reason_summary["period"].map({
        "Before LSS": 62,
        "After LSS": 43
    })
)

stop_reason_summary["downtime_min_per_day"] = (
    stop_reason_summary["total_downtime_min"]
    / stop_reason_summary["production_days"]
)

stop_reason_summary = stop_reason_summary.sort_values(
    ["period", "downtime_min_per_day"],
    ascending=[True, False]
)

stop_reason_summary

,period,stop_group,stop,events,total_downtime_min,avg_duration_min,production_days,downtime_min_per_day
1,After LSS,FAILURE,MECHANICAL FAILURE,35,818.0,23.371429,43,19.023256
7,After LSS,MATERIALS,MANUAL BRICK HANDLING,80,766.0,9.575000,43,17.813953
10,After LSS,MATERIALS,RACK STARVATION,103,765.0,7.427184,43,17.790698
0,After LSS,FAILURE,AUTOMATION FAILURE,23,429.0,18.652174,43,9.976744
2,After LSS,FAILURE,OTHER FAILURE,35,334.0,9.542857,43,7.767442
11,After LSS,MATERIALS,RACKS,33,258.0,7.818182,43,6.000000
6,After LSS,MATERIALS,KILN CAR STARVATION,12,133.0,11.083333,43,3.093023
3,After LSS,FAILURE,WIRE FAILURE,23,109.0,4.739130,43,2.534884
5,After LSS,MATERIALS,EXTRUSION DEFECTS,12,84.0,7.000000,43,1.953488
12,After LSS,MATERIALS,WET WIP BRICKS,2,60.0,30.000000,43,1.395349


In [12]:
stop_comparison = (
    stop_reason_summary
    .pivot_table(
        index=["stop_group", "stop"],
        columns="period",
        values="downtime_min_per_day",
        fill_value=0
    )
    .reset_index()
)

stop_comparison["change_min_per_day"] = (
    stop_comparison["After LSS"]
    - stop_comparison["Before LSS"]
)

stop_comparison = stop_comparison.sort_values(
    "change_min_per_day",
    ascending=False
)

stop_comparison

period,stop_group,stop,After LSS,Before LSS,change_min_per_day
11,MATERIALS,RACK STARVATION,17.790698,1.456452,16.334246
2,FAILURE,OTHER FAILURE,7.767442,0.311290,7.456152
3,FAILURE,WIRE FAILURE,2.534884,0.000000,2.534884
12,MATERIALS,RACKS,6.000000,3.700000,2.300000
6,MATERIALS,EXTRUSION DEFECTS,1.953488,0.000000,1.953488
14,MATERIALS,WET WIP BRICKS,1.395349,0.000000,1.395349
5,MATERIALS,CRACKED WIP BRICKS,0.930233,0.000000,0.930233
7,MATERIALS,KILN CAR STARVATION,3.093023,2.330645,0.762378
10,MATERIALS,METAL DETECTOR,0.418605,0.175806,0.242798
4,MATERIALS,AUTOMATION FAILURE,0.000000,0.061290,-0.061290


### Downtime Findings

LSS was associated with substantial reductions in Automation Failure, Manual Brick Handling, and Mechanical Failure downtime.

However, the loss profile shifted after LSS. Rack Starvation emerged as one of the largest downtime contributors, while Other Failure also increased.

This suggests that the original major losses were reduced, but material-flow and newly emerging failure modes became the next improvement priorities.